# S1-02: 멀티턴 대화와 시스템 프롬프트
**Skilljar L06-L09: Multi-Turn Conversations / Chat Exercise / System Prompts**

## 학습 목표
- Claude API의 무상태(Stateless) 특성을 이해한다
- 헬퍼 함수로 멀티턴 대화를 구현한다
- 시스템 프롬프트로 Claude의 역할과 행동을 정의한다
- 건축공학 도메인 전문 어시스턴트를 설계한다

In [ ]:
# 패키지 설치
%pip install anthropic python-dotenv

In [ ]:
# 환경변수 로드
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# 클라이언트 생성
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

## 1. Claude는 무상태(Stateless)

Claude API는 **이전 대화를 기억하지 못한다**. 매 API 호출은 완전히 독립적이다.

대화를 이어가려면 **개발자가 직접 대화 이력(messages 리스트)을 관리**해야 한다.

In [ ]:
# 무상태 문제 시연: 두 번째 호출이 첫 번째를 기억하지 못함

# 1번째 호출
msg1 = client.messages.create(
    model=model,
    max_tokens=200,
    messages=[{"role": "user", "content": "RC 보의 최소 철근비에 대해 설명해줘."}]
)
print("=== 호출 1 ===")
print(msg1.content[0].text[:200], "...")

# 2번째 호출 — 이전 맥락이 없음
msg2 = client.messages.create(
    model=model,
    max_tokens=200,
    messages=[{"role": "user", "content": "그 값을 fck=30MPa일 때 계산해줘."}]
)
print("\n=== 호출 2 (맥락 없음 — 무엇의 값인지 모름) ===")
print(msg2.content[0].text[:200], "...")

## 2. 헬퍼 함수 정의

대화 이력 관리를 단순화하기 위한 3가지 헬퍼 함수:

```
messages = []
    |
add_user_message()       <-- 사용자 입력 추가
    |
chat(messages)           <-- 전체 이력 전송 -> 응답 수신
    |
add_assistant_message()  <-- 응답을 이력에 추가
    |
(반복)
```

In [ ]:
# 헬퍼 함수 정의

def add_user_message(messages: list, text: str):
    """사용자 메시지를 대화 이력에 추가"""
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages: list, text: str):
    """어시스턴트 응답을 대화 이력에 추가"""
    messages.append({"role": "assistant", "content": text})

def chat(messages: list, system: str = None) -> str:
    """대화 이력을 전송하고 응답 텍스트를 반환"""
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages
    }
    if system:
        params["system"] = system
    response = client.messages.create(**params)
    return response.content[0].text

print("헬퍼 함수 정의 완료")

## 3. 멀티턴 대화 구현

핵심: 사용자 메시지와 어시스턴트 응답을 **모두 리스트에 누적**해서 매 호출마다 전체 이력을 전송한다.

In [ ]:
# 멀티턴 대화 예시 — 이전 맥락을 유지하며 대화
messages = []

# 1턴: 사용자 질문
add_user_message(messages, "RC 보의 설계에서 최소 철근비의 의미가 뭐야?")
response = chat(messages)
add_assistant_message(messages, response)
print("=== 1턴 ===")
print(response)

# 2턴: 후속 질문 (이전 맥락 활용)
add_user_message(messages, "그렇다면 KDS 기준에서 최소 철근비 산정 공식은?")
response = chat(messages)
add_assistant_message(messages, response)
print("\n=== 2턴 ===")
print(response)

# 3턴: 실제 계산 요청
add_user_message(messages, "fck=30MPa, fy=400MPa일 때 실제로 계산해줘.")
response = chat(messages)
add_assistant_message(messages, response)
print("\n=== 3턴 ===")
print(response)

In [ ]:
# 대화 이력 확인 — 어떤 메시지들이 쌓여 있는지 확인
for i, msg in enumerate(messages):
    role = msg["role"]
    content_preview = msg["content"][:80] + "..." if len(msg["content"]) > 80 else msg["content"]
    print(f"[{i}] {role}: {content_preview}")

## 4. 시스템 프롬프트 (System Prompts)

시스템 프롬프트는 Claude의 **역할, 톤, 행동 규칙**을 정의하는 특별한 지시이다.

| 구분 | 시스템 프롬프트 (`system=`) | 사용자 메시지 (`messages`) |
|---|---|---|
| **위치** | `system` 매개변수 (별도) | `messages` 리스트 내부 |
| **역할** | Claude의 인격/행동 정의 | 실제 대화 내용 |
| **지속성** | 모든 턴에 일관 적용 | 턴마다 변화 |
| **비유** | 직원 채용 시 직무기술서 (JD) | 일상적인 업무 지시 |

In [ ]:
# 시스템 프롬프트 예시: 수학 튜터
# 직접 답을 주지 않고 힌트로 유도하는 역할을 부여

system_tutor = """당신은 인내심 있는 수학 튜터입니다.

행동 규칙:
- 학생의 질문에 직접 답을 주지 마세요
- 힌트와 가이드 질문으로 단계별로 유도하세요
- 학생이 스스로 답을 찾도록 도와주세요
- 격려하는 톤을 유지하세요
"""

messages = []
add_user_message(messages, "3x + 7 = 22 를 풀어줘")
response = chat(messages, system=system_tutor)
print("=== 수학 튜터 (답을 직접 주지 않음) ===")
print(response)

In [ ]:
# 비교: 시스템 프롬프트 없이 같은 질문
messages_no_system = []
add_user_message(messages_no_system, "3x + 7 = 22 를 풀어줘")
response_no_system = chat(messages_no_system)
print("=== 시스템 프롬프트 없음 (바로 답을 줌) ===")
print(response_no_system)

## 5. 시스템 프롬프트 + 멀티턴 조합

시스템 프롬프트는 **모든 턴에 일관 적용**된다. `chat()` 호출마다 동일한 `system`을 전달해야 한다.

In [ ]:
# 시스템 프롬프트 + 멀티턴: 구조역학 튜터

system_mechanics = """당신은 구조역학 교수입니다.

행동 규칙:
- 질문에 대해 먼저 핵심 개념을 설명하세요
- 그 다음 공식을 제시하세요
- 마지막으로 간단한 예제를 보여주세요
- 한국어로 답변하되, 전문 용어는 영문 병기하세요
"""

messages = []

# 1턴
add_user_message(messages, "단순보의 최대 처짐을 구하는 방법을 설명해줘.")
r1 = chat(messages, system=system_mechanics)
add_assistant_message(messages, r1)
print("=== 1턴 ===")
print(r1)

# 2턴 — 이전 맥락을 활용한 후속 질문
add_user_message(messages, "L=6m, w=20kN/m, E=200GPa, I=5000cm4일 때 값을 계산해줘.")
r2 = chat(messages, system=system_mechanics)
add_assistant_message(messages, r2)
print("\n=== 2턴 ===")
print(r2)

---
## 건축공학 실습 과제

### 과제: KDS 기반 구조 검토 시스템 프롬프트 설계

아래 요구사항을 만족하는 시스템 프롬프트를 설계하고, 멀티턴 대화로 구조 검토를 수행하세요.

**시스템 프롬프트 요구사항:**
1. KDS 콘크리트구조 설계기준(KDS 14 20 00)에 정통한 전문가 역할
2. 모든 검토에 적용 KDS 조항 번호를 명시
3. 계산 과정을 단계별로 표시
4. 결과를 표 형식으로 정리
5. 부적합 시 개선 방안 제안

**멀티턴 대화 시나리오:**
- 1턴: RC 기둥 설계 조건 제시 및 검토 요청
- 2턴: 내진등급 변경에 따른 재검토
- 3턴: 부적합 항목에 대한 개선안 요청

In [ ]:
# TODO: 시스템 프롬프트를 설계하세요
structural_system = """당신은 한국 건축구조 설계기준(KDS)에 정통한 구조공학 전문 AI 어시스턴트입니다.

전문 분야:
- 콘크리트구조 설계기준 (KDS 14 20 00)
- 내진설계 기준 (KDS 41 17 00)
- 하중 기준 (KDS 41 10 15)

행동 규칙:
1. 모든 검토에 적용 KDS 조항 번호를 명시하세요
2. 계산 과정을 단계별로 보여주세요
3. 결과를 검토항목별 표로 정리하세요
4. 설계 부적합 시 개선 방안을 제안하세요
5. 불확실한 가정은 명시적으로 표기하세요
"""

messages = []

# 1턴: 초기 설계 조건 검토 요청
add_user_message(messages, """다음 RC 기둥의 설계 적정성을 검토해주세요.

- 기둥 단면: 500mm x 500mm
- 콘크리트 강도 (fck): 24 MPa
- 주근: 8-D25 (SD400)
- 설계 축력 (Pu): 2,500 kN
- 설계 모멘트 (Mu): 150 kN-m
- 내진등급: 일반 (내진설계범주 C)""")

response1 = chat(messages, system=structural_system)
add_assistant_message(messages, response1)
print("=== 1턴: 초기 설계 검토 ===")
print(response1)

In [ ]:
# 2턴: 내진등급 변경에 따른 재검토
add_user_message(messages, """내진등급이 '특등급 (내진설계범주 D)'으로 변경되었습니다.
변경된 조건에서 기존 설계가 여전히 적합한지 재검토해주세요.""")

response2 = chat(messages, system=structural_system)
add_assistant_message(messages, response2)
print("=== 2턴: 내진등급 변경 재검토 ===")
print(response2)

In [ ]:
# 3턴: 개선안 요청
add_user_message(messages, "부적합 항목에 대한 개선안을 제시해주세요.")

response3 = chat(messages, system=structural_system)
add_assistant_message(messages, response3)
print("=== 3턴: 개선안 ===")
print(response3)